In [0]:
%pip install pyproj --quiet
dbutils.library.restartPython()

In [0]:
# 02c — Silver: Índice de directividad de recorridos
import pandas as pd
from pyproj import Geod
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_BRONZE = "a_bronze"
SCHEMA_SILVER = "b_silver"
UMBRAL_CIRCULAR_METROS = 200

In [0]:
## 1. Lectura y mapeo shape_id → route_id

shapes_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_shapes").toPandas()
trips_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_trips").toPandas()
routes_pdf = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_routes").toPandas()

print(f"Puntos de shapes: {len(shapes_pdf)} | shapes distintos: {shapes_pdf['shape_id'].nunique()}")

if "shape_id" not in trips_pdf.columns:
    raise ValueError("trips.txt no trae columna shape_id — no se puede vincular shapes a líneas.")

shape_route = trips_pdf[["shape_id", "route_id"]].drop_duplicates().dropna(subset=["shape_id"])
shape_route = shape_route.merge(
    routes_pdf[["route_id", "route_short_name", "route_long_name"]], on="route_id", how="left"
)

print(f"Pares shape_id-route_id únicos: {len(shape_route)}")

In [0]:
## 2. Cálculo geodésico de longitud y distancia recta por shape

geod = Geod(ellps="WGS84")

def calcular_metricas_shape(puntos_ordenados):
    """Recibe un DataFrame de un solo shape_id, ya ordenado por shape_pt_sequence."""
    lons = puntos_ordenados["shape_pt_lon"].tolist()
    lats = puntos_ordenados["shape_pt_lat"].tolist()

    if len(lons) < 2:
        return None, None

    longitud_m = geod.line_length(lons, lats)
    _, _, distancia_recta_m = geod.inv(lons[0], lats[0], lons[-1], lats[-1])
    return longitud_m, distancia_recta_m


resultados = []
for shape_id, grupo in shapes_pdf.sort_values("shape_pt_sequence").groupby("shape_id"):
    longitud_m, distancia_recta_m = calcular_metricas_shape(grupo)
    resultados.append({"shape_id": shape_id, "longitud_m": longitud_m, "distancia_recta_m": distancia_recta_m})

metricas_pdf = pd.DataFrame(resultados)

antes = len(metricas_pdf)
metricas_pdf = metricas_pdf.dropna(subset=["longitud_m", "distancia_recta_m"])
print(f"Shapes descartados por tener menos de 2 puntos: {antes - len(metricas_pdf)}")

metricas_pdf["es_circular"] = metricas_pdf["distancia_recta_m"] < UMBRAL_CIRCULAR_METROS
metricas_pdf["ratio_directividad"] = metricas_pdf.apply(
    lambda r: r["longitud_m"] / r["distancia_recta_m"] if not r["es_circular"] else None, axis=1
)

print(f"Shapes circulares (distancia recta < {UMBRAL_CIRCULAR_METROS}m): {metricas_pdf['es_circular'].sum()}")
print(metricas_pdf["ratio_directividad"].describe())

In [0]:
## 3. Armado final y escritura

route_directness = shape_route.merge(metricas_pdf, on="shape_id", how="inner")
route_directness["longitud_km"] = route_directness["longitud_m"] / 1000
route_directness["distancia_recta_km"] = route_directness["distancia_recta_m"] / 1000
route_directness = route_directness.drop(columns=["longitud_m", "distancia_recta_m"])
route_directness["_processed_at"] = datetime.now(timezone.utc)

print(f"Filas finales: {len(route_directness)}")
route_directness.sort_values("ratio_directividad", ascending=False).head(10)

spark.createDataFrame(route_directness).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.route_directness")

count = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.route_directness").count()
print(f"route_directness: {count} filas escritas.")